# Monthly data on deposits and casualties


In [28]:
import pandas as pd
import statsmodels.formula.api as smf


In [11]:
# load marriages (cross-sectional)
data_cs = pd.read_csv('intermediate/war_data.csv')
data_cs = data_cs[["Region", "Excessive Marriages per100k"]]

#load soc-econ data
data_se = pd.read_csv('intermediate/soc_econ_data.csv')

#load monthly casualties 
df_monthly = pd.read_csv('data/daily.csv')
df_monthly = df_monthly[['region', 'branch', 'name','slavic','death_month']]
df_monthly = df_monthly.rename(columns={df_monthly.columns[0] : "Region"})

#load monthly deposits
dep = pd.read_excel("data/deposits.xlsx", header = 1)[:95]
dep = dep.rename(columns={dep.columns[0] : "Region"})
dep_long = pd.melt(dep, id_vars=['Region'],
                   var_name='Date', 
                   value_name='Deposits') 



In [12]:
region_name_mapping = {
    'Архангельская область без данных по Ненецкому автономному округу': 'Архангельская область без АО',
    'в том числе Ненецкий автономный округ': 'Ненецкий АО',
    'Кемеровская область - Кузбасс': 'Кемеровская область',
    'Город Москва столица Российской Федерации город федерального значения': 'Москва',
    'Город Санкт-Петербург город федерального значения': 'Санкт-Петербург',
    'Город федерального значения Севастополь': 'Севастополь',
    'Еврейская автономная область': 'Еврейская АО',
    'Кабардино-Балкарская Республика': 'Кабардино-Балкария',
    'Карачаево-Черкесская Республика': 'Карачаево-Черкесия',
    'Республика Адыгея (Адыгея)': 'Республика Адыгея',
    'Республика Саха (Якутия)': 'Якутия',
    'Республика Северная Осетия - Алания': 'Северная Осетия',
    'Республика Татарстан (Татарстан)': 'Республика Татарстан',
    'Чувашская Республика - Чувашия': 'Чувашская Республика',
    'Тюменская область без данных по Ханты-Мансийскому автономному округу - Югре и Ямало-Ненецкому автономному округу': 'Тюменская область без АО',
    'в том числе Ханты-Мансийский автономный округ - Югра': 'Ханты-Мансийский АО',
    'в том числе Ямало-Ненецкий автономный округ': 'Ямало-Hенецкий АО',
    'Чукотский автономный округ': 'Чукотский АО',
    'г. Москва' : 'Москва',
    'г. Санкт-Петербург' : 'Санкт-Петербург',
    'г. Севастополь' : 'Севастополь'
}

dep_long['Region'] = dep_long['Region'].replace(region_name_mapping)



In [13]:
region_name_mapping = {
    'Архангельская область': 'Архангельская область без АО',
    'Ненецкий автономный округ': 'Ненецкий АО',
    'Еврейская автономная область': 'Еврейская АО',
    'Кабардино-Балкарская Республика': 'Кабардино-Балкария',
    'Республика Карачаево-Черкесия': 'Карачаево-Черкесия',
    'Республика Саха (Якутия)': 'Якутия',
    'Республика Северная Осетия-Алания': 'Северная Осетия',
    'Тюменская область': 'Тюменская область без АО',
    'Ханты-Мансийский автономный округ - Югра': 'Ханты-Мансийский АО',
    'Ямало-Ненецкий автономный округ': 'Ямало-Hенецкий АО',
    'Чукотский автономный округ': 'Чукотский АО',
}

df_monthly['Region'] = df_monthly['Region'].replace(region_name_mapping)


In [14]:
df_monthly['death_month'] = pd.to_datetime(df_monthly['death_month'].astype(str), format='%m.%Y', errors ='coerce')
dep_long['Date'] = pd.to_datetime(dep_long['Date'].astype(str), format='%d.%m.%Y', errors ='coerce')

df_agg = df_monthly.dropna(subset = ['death_month']).groupby(['Region', 'death_month']).agg(
    slavic_name = ('slavic', lambda x: (x == 1).sum()),
    non_slavic_name = ('slavic', lambda x: (x == 0).sum())).reset_index() 



In [15]:
all_regions = df_monthly['Region'].unique()
all_months = pd.date_range(
    start=df_monthly['death_month'].min(),
    end=df_monthly['death_month'].max(),
    freq='MS' 
)

complete_grid = pd.MultiIndex.from_product(
    [all_regions, all_months],
    names=['Region', 'death_month']
).to_frame(index=False)


In [16]:
final_result = (
    complete_grid.merge(
        df_agg,
        on=['Region', 'death_month'],
        how='left'
    )
    .fillna({'slavic_name': 0, 'non_slavic_name': 0})  
    .sort_values(['Region', 'death_month'])
)
final_result[['slavic_cumulative', 'non_slavic_cumulative']] = (
    final_result.groupby('Region')[['slavic_name', 'non_slavic_name']]
    .cumsum()
)

final_result["total"] = final_result['slavic_cumulative'] + final_result['non_slavic_cumulative']
final_result['slavic_name %'] = final_result['slavic_cumulative'] / final_result["total"]
final_result.columns.values[1] = 'Date'

In [19]:
final_result = final_result.merge(dep_long,
                                  on = ['Region', 'Date'],
                                  how = 'left')
final_result = final_result.merge(data_cs,
                                  on = 'Region',
                                  how = 'left')
final_result = final_result.merge(data_se,
                                  on = 'Region',
                                  how = 'left')

final_result["treat"] = (final_result["Date"] > "2022-09-21 00:00:00").astype('int')

final_result

,Region,Date,slavic_name,non_slavic_name,slavic_cumulative,non_slavic_cumulative,total,slavic_name %,Deposits,Excessive Marriages per100k,Unnamed: 0,% Russians,unemp,median_income,share_poverty
0,Алтайский край,2022-02-01,1.0,0.0,1.0,0.0,1.0,1.000000,223441,343.285700,70.0,95.45,5.5,20783.2,16.5
1,Алтайский край,2022-03-01,33.0,2.0,34.0,2.0,36.0,0.944444,221270,343.285700,70.0,95.45,5.5,20783.2,16.5
2,Алтайский край,2022-04-01,30.0,2.0,64.0,4.0,68.0,0.941176,227749,343.285700,70.0,95.45,5.5,20783.2,16.5
3,Алтайский край,2022-05-01,57.0,5.0,121.0,9.0,130.0,0.930769,238135,343.285700,70.0,95.45,5.5,20783.2,16.5
4,Алтайский край,2022-06-01,31.0,3.0,152.0,12.0,164.0,0.926829,241712,343.285700,70.0,95.45,5.5,20783.2,16.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3733,Ярославская область,2025-03-01,4.0,1.0,492.0,16.0,508.0,0.968504,416577,232.629579,17.0,96.52,5.9,26858.2,8.9
3734,Ярославская область,2025-04-01,0.0,0.0,492.0,16.0,508.0,0.968504,424381,232.629579,17.0,96.52,5.9,26858.2,8.9
3735,Ярославская область,2025-05-01,0.0,0.0,492.0,16.0,508.0,0.968504,434617,232.629579,17.0,96.52,5.9,26858.2,8.9
3736,Ярославская область,2025-06-01,0.0,0.0,492.0,16.0,508.0,0.968504,NaN,232.629579,17.0,96.52,5.9,26858.2,8.9


In [89]:
df = final_result
df['excess_x_treat'] = df['Excessive Marriages per100k'] * df['treat']
df = df.rename(columns={'Excessive Marriages per100k': 'excess_marriage'})
df = df.rename(columns={'% Russians': 'share_rus'})

df_model = df[["Region", 'Date', "Deposits", "excess_x_treat", "treat", "excess_marriage", "total", 
               "unemp", "median_income", "share_poverty", "share_rus"]]
df_model = df_model.replace([np.inf, -np.inf], np.nan).dropna()

model1 = smf.ols(
    formula='Deposits ~ total + unemp + median_income + share_poverty + share_rus + excess_marriage + treat + excess_x_treat',
    data=df_model
).fit()

model1.summary()


C:\Users\Robert_Beta\AppData\Local\Temp\ipykernel_9208\1870795876.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_model = df_model.replace([np.inf, -np.inf], np.nan).dropna()


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:               Deposits   R-squared:                       0.231
Model:                            OLS   Adj. R-squared:                  0.229
Method:                 Least Squares   F-statistic:                     126.0
Date:                 Сб, 26 июл 2025   Prob (F-statistic):          3.94e-185
Time:                        23:23:00   Log-Likelihood:                -52444.
No. Observations:                3360   AIC:                         1.049e+05
Df Residuals:                    3351   BIC:                         1.050e+05
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
===================================================================================
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept       -2.778e+06   2.87e+05     -9.682      0.000   -3.34e+06   -2.22e+06
total             532.8345     41.304     12.900      0.000     451.850     613.819
unemp           -6794.8077   1.13e+04     -0.603      0.546   -2.89e+04    1.53e+04
median_income      80.3276      3.455     23.250      0.000      73.554      87.102
share_poverty    4.937e+04   1.14e+04      4.335      0.000     2.7e+04    7.17e+04
share_rus        8642.0270   1645.405      5.252      0.000    5415.928    1.19e+04
excess_marriage -1563.2344    339.845     -4.600      0.000   -2229.560    -896.909
treat           -1.489e+05   1.14e+05     -1.312      0.190   -3.72e+05    7.37e+04
excess_x_treat   -113.5786    347.810     -0.327      0.744    -795.519     568.362
==============================================================================
Omnibus:                     4198.672   Durbin-Watson:                   0.064
Prob(Omnibus):                  0.000   Jarque-Bera (JB):           638904.436
Skew:                           6.772   Prob(JB):                         0.00
Kurtosis:                      69.183   Cond. No.                     3.44e+05
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 3.44e+05. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [56]:
df_model.Region.unique()

array(['Алтайский край', 'Амурская область',
       'Архангельская область без АО', 'Астраханская область',
       'Белгородская область', 'Брянская область', 'Владимирская область',
       'Волгоградская область', 'Вологодская область',
       'Воронежская область', 'Еврейская АО', 'Забайкальский край',
       'Ивановская область', 'Иркутская область', 'Кабардино-Балкария',
       'Калининградская область', 'Калужская область', 'Камчатский край',
       'Карачаево-Черкесия', 'Кемеровская область', 'Кировская область',
       'Костромская область', 'Краснодарский край', 'Красноярский край',
       'Курганская область', 'Курская область', 'Ленинградская область',
       'Липецкая область', 'Магаданская область', 'Москва',
       'Московская область', 'Мурманская область', 'Ненецкий АО',
       'Нижегородская область', 'Новгородская область',
       'Новосибирская область', 'Омская область', 'Оренбургская область',
       'Орловская область', 'Пензенская область', 'Пермский край',
      

'Excessive Marriages per100k'